# tables and csv files

The `pandas.DataFrame` object stores a table.

Tables are important in SQL (relational databases) and languages designed for data analysis. `pandas.DataFrame` is similar to the `data.frame()` function in R and the `DataFrame` type in Spark.

BONUS PROBLEM FOR BORED PEOPLE: Look up _column-oriented database_ to learn more about how `pandas` stores data.

In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
HOLIDAYS_PATH = Path.cwd() / "data/holidays.csv"

## create a table

In this table, each row is a holiday from the [New York Stock Exchange 2026 calendar]. Each holiday has 3 _features_: date, English name, and 中文 name.

**Caution:** These are automated translations. They might be wrong.

[New York Stock Exchange 2026 calendar]: https://www.nyse.com/trade/hours-calendars

In [ ]:
# There are many ways to store tables. (Many of them are bad.)
# This is one of the simplest: it's a list of tuples.
# This is similar to how SQL databases store data.

holidays = [
    ("2026-01-01", "New Year's Day", "元旦"),
    ("2026-12-25", "Christmas Day", "圣诞节"),
    ("2026-06-19", "Juneteenth", "六月节"),
    ("2026-11-26", "Thanksgiving Day", "感恩节"),
    ("2026-07-03", "Independence Day", "独立日"),
    ("2026-01-19", "Martin Luther King, Jr. Day", "马丁路德金纪念日"),
    ("2026-04-03", "Good Friday", "耶稣受难日"),
    ("2026-05-25", "Memorial Day", "纪念日"),
    ("2026-09-07", "Labor Day", "劳动节"),
    ("2026-02-16", "Washington's Birthday", "华盛顿生日"),
]

# Show each holiday and check if they look OK
for date, english, 中文 in holidays:
    print(f"{date} is {english} ({中文})")

In [ ]:
# Call the DataFrame() constructor.
data = pd.DataFrame(holidays)

# My raw data has no column names. Let's fix that.
data.columns = "date english 中文".split()

# Use Jupyter to display the table.
data

## check data types and nulls

In [ ]:
# Each column has a type.
data.dtypes

In [ ]:
# I always write dates like this: 2024-04-01
# It's similar to 2024年4月1号, but I always include
# leading zeros like the 0 in 04 and 0 in 01.
# If I lexsort these strings, their order will be correct.

sorted(data['date'])

In [ ]:
# Use pandas to convert my string dates to pd.Timestamp type.
data['date'] = pd.to_datetime(data['date'])
data.dtypes

In [ ]:
# Check for null markers. If pandas can't convert a date,
# it will use NaT (Not a Timestamp) to mark it null.
data.isnull()

## set index and sort

**BONUS PROBLEM FOR BORED PEOPLE:**
Read about _SQL index columns_. It's the same idea. Index columns are optimized for fast selections and filtering.

In [ ]:
# Use the 'date' column as an Index and sort it.
# .sort_index(axis=1) also sorts column names.
data = (
    data
    .set_index('date')
    .sort_index()
    .sort_index(axis=1)
)
data

In [ ]:
# Like a dictionary, DataFrame has keys and values.
# The keys are (non-Index) column names.
data.keys()

In [ ]:
# The values are a 2D NumPy array.
data.values

In [ ]:
# The index is a special type of object.
data.index

In [ ]:
# The index also uses NumPy arrays.
data.index.values

## remove the index

In [ ]:
# This returns a new table with no index columns.
data.reset_index()

## DataFrame is a key-value store

In [ ]:
# Use dictionary key syntax to select a column.
data['english']

In [ ]:
# Each column is a pandas.Series object.
type(data['english'])

## Series is a key-value store

In [ ]:
ser = data['english']
ser.keys()

In [ ]:
ser.values

## filter the table

**BONUS PROBLEM FOR BORED PEOPLE:**
How would you do this in SQL? It's something like `SELECT {columns} FROM {table} WHERE {conditions}`.

In [ ]:
# DateTimeIndex can do fancy date and time selection.
data.loc['2026-01']

In [ ]:
data.loc['2026-01':'2026-04']

In [ ]:
data.loc['2026-01':'2026-04', '中文']

In [ ]:
# We can also filter by non-index columns.
data.loc[data['中文'].str.endswith('日')]

In [ ]:
# To get 1 value, use .at[] instead of .loc[].
data.at['2026-01-01', '中文']

## iterate over a DataFrame

In [ ]:
# This will only give us column names.
list(data)

In [ ]:
# Like a dictionary, DataFrame has an .items() method.
# Each key is a column name. Each value is a pandas.Series.
for key, val in data.items():
    print(f"{key} : {type(val)}")

In [ ]:
# Iterate over rows with .itertuples().
for row in data.itertuples():
    print(row)

## serialize a table as a JSON string

**Caution:** Some JSON formats discard index columns. If we want to keep the index, we might need to `.reset_index()` before serializing.

In [ ]:
kw = {
    "indent": 2,
    "force_ascii": False,
}

In [ ]:
print(data.to_json(**kw))

In [ ]:
print(data.to_json(**kw, orient='records'))

In [ ]:
print(data.to_json(**kw, orient='split'))

In [ ]:
print(data.to_json(**kw, orient='table'))

## save a DataFrame as a CSV

**Caution:** If you save a table to CSV, then read it, some of the datatypes might change. To preserve type, we need files like `.hdf5` or `.parquet`.

**BONUS PROBLEM FOR BORED PEOPLE:**
Wes McKinney (the author of pandas) is working on this problem. Look up `pyarrow` and `Apache Parquet`.

In [ ]:
print(f"Save table to {HOLIDAYS_PATH}")
data.to_csv(HOLIDAYS_PATH)

In [ ]:
print(f"Read table from {HOLIDAYS_PATH}")
csvdata = pd.read_csv(HOLIDAYS_PATH)
csvdata.dtypes